# ASCII Plots Tutorial

This tutorial demonstrates the text-based plotting functions in `balance`.
ASCII plots are useful in terminals, CI logs, notebooks with limited rendering,
or anywhere you want a quick visual comparison without a graphical backend.

We cover:
1. **Grouped barplots** (`ascii_plot_bar`) for categorical variables
2. **Grouped histograms** (`ascii_plot_hist`) for numeric variables
3. **Comparative histograms** (`ascii_comparative_hist`) with baseline-relative rendering
4. **`ascii_plot_dist`** — the all-in-one dispatcher
5. Using `library="balance"` with the `.covars().plot()` API

## Setup

Load the toy dataset and run an IPW adjustment so we have three datasets
to compare: unadjusted sample, adjusted sample, and target population.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from balance import Sample, load_data

target_df, sample_df = load_data()

sample = Sample.from_frame(sample_df, outcome_columns=["happiness"])
target = Sample.from_frame(target_df, outcome_columns=["happiness"])
sample_with_target = sample.set_target(target)
adjusted = sample_with_target.adjust()

## 1. All covariates at a glance

The easiest way to get an ASCII comparison is through the standard `.covars().plot()`
API with `library="balance"`. This automatically classifies each variable as
categorical or numeric and renders grouped barplots or histograms accordingly.

The three fill characters distinguish the datasets:
- `█` = **sample** (unadjusted)
- `▒` = **adjusted** (after IPW)
- `▐` = **population** (target)

In [ ]:
adjusted.covars().plot(library="balance");

## 2. Category spacing

By default, barplots insert a blank line between categories for readability.
You can disable this with `separate_categories=False` to get a more compact view.

In [ ]:
# Compact view (no blank lines between categories)
adjusted.covars().plot(
    library="balance", variables=["gender"],
    separate_categories=False,
);

In [ ]:
# Default view (blank lines between categories)
adjusted.covars().plot(
    library="balance", variables=["gender"],
);

## 3. Focusing on a single variable

Pass `variables=["income"]` to plot just one covariate. For numeric variables
the output is a grouped histogram where each bin shows the weighted proportion
for the unadjusted sample, adjusted sample, and target population.

In [ ]:
adjusted.covars().plot(
    library="balance", variables=["income"],
);

## 4. Grouped histogram (`ascii_plot_hist`)

`ascii_plot_hist` provides a side-by-side view: each dataset gets its own bar
(with a distinct fill character) so you can compare how mass is distributed
across bins. Bar lengths are proportional to weighted frequency within each dataset.

In [ ]:
from balance.stats_and_plots.ascii_plots import ascii_plot_hist

dfs = [
    {"df": target_df, "weight": None},
    {"df": sample_df, "weight": None},
]
print(ascii_plot_hist(
    dfs, names=["Target", "Sample"], column="income",
))

## 5. Comparative histogram (`ascii_comparative_hist`)

When you want to see *how* a dataset differs from a baseline,
`ascii_comparative_hist` renders each bin relative to the first dataset:

- `█` (solid) = the proportion shared with the baseline
- `▒` (shaded) = excess beyond the baseline
- `   ]` (right bracket) = deficit relative to the baseline

In [ ]:
from balance.stats_and_plots.ascii_plots import ascii_comparative_hist

# Target (baseline) vs unadjusted sample
dfs = [
    {"df": target_df, "weight": None},
    {"df": sample_df, "weight": None},
]
print(ascii_comparative_hist(
    dfs, names=["Target", "Sample"], column="income",
))

### Three-way comparative histogram

You can also compare the target, unadjusted, and adjusted distributions
side by side. Here the target is the baseline, so `▒` shows where the
other datasets have **more** mass and `   ]` shows where they have **less**.

In [ ]:
dfs = [
    {"df": target.covars().df, "weight": target.weight_column},
    {"df": sample_with_target.covars().df, "weight": sample_with_target.weight_column},
    {"df": adjusted.covars().df, "weight": adjusted.weight_column},
]
print(ascii_comparative_hist(
    dfs, names=["Target", "Unadjusted", "Adjusted"],
    column="income",
))